
# Week 2 — Semantic Retrieval Pipeline

This notebook implements the Week 2 requirements for the RAG system using the processed chunk metadata file generated in Week 1.

## Scope covered
- Embedding generation pipeline
- Vector database setup with FAISS
- Similarity search workflow
- Metadata-based retrieval filtering
- Retrieval evaluation and latency analysis

This notebook stays within the Week 2 instructions and does not include LLM integration, prompt engineering, or deployment.



## Embedding Model Justification

**Selected model:** `sentence-transformers/all-MiniLM-L6-v2`

Why this model is appropriate for Week 2:
- It is widely used for semantic similarity retrieval tasks.
- It is lightweight enough for local execution in notebook environments.
- It produces fixed-size dense vectors suitable for FAISS indexing.
- It is practical for validating embedding generation, retrieval quality, and latency before moving to LLM integration.

The goal in Week 2 is reliable semantic retrieval, so a compact sentence-transformer model is a suitable baseline.


In [1]:

!pip install -q sentence-transformers faiss-cpu scikit-learn pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.7 MB/s eta 0:00:00


In [3]:

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

CONFIG = {
    "metadata_path_candidates": [
        Path("metadata.csv"),
        Path("/content/metadata.csv")
    ],
    "text_column": "chunk",
    "id_column": "chunk_id",
    "metadata_columns": [
        "department", "document_type", "category", "region", "year", "source", "file_name"
    ],
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "vectorstore_dir": Path("week2_vectorstore"),
    "top_k": 5,
    "normalize_embeddings": True,
    "evaluation_sample_size": 100,
    "random_state": 42
}

CONFIG["vectorstore_dir"].mkdir(parents=True, exist_ok=True)

metadata_path = None
for candidate in CONFIG["metadata_path_candidates"]:
    if candidate.exists():
        metadata_path = candidate
        break

if metadata_path is None:
    raise FileNotFoundError("metadata.csv not found in expected locations.")

print(f"Using metadata file: {metadata_path}")


Using metadata file: metadata.csv


In [4]:

df = pd.read_csv(metadata_path)

required_columns = [CONFIG["id_column"], CONFIG["text_column"]] + CONFIG["metadata_columns"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# basic validation required before embedding generation
initial_rows = len(df)
df = df.dropna(subset=[CONFIG["text_column"], CONFIG["id_column"]]).copy()
df[CONFIG["text_column"]] = df[CONFIG["text_column"]].astype(str).str.strip()
df = df[df[CONFIG["text_column"]] != ""].reset_index(drop=True)

print("Rows before validation:", initial_rows)
print("Rows after validation:", len(df))
print("Columns available:", list(df.columns))
print(df.head(3))


Rows before validation: 2129
Rows after validation: 2129
Columns available: ['chunk_id', 'chunk', 'chunk_tokens', 'department', 'document_type', 'category', 'region', 'year', 'source', 'file_name']
            chunk_id                                              chunk  \
0  ag_news_0_chunk_1  Wall St. Bears Claw Back Into the Black (Reute...   
1  ag_news_1_chunk_1  Carlyle Looks Toward Commercial Aerospace (Reu...   
2  ag_news_2_chunk_1  Oil and Economy Cloud Stocks' Outlook (Reuters...   

   chunk_tokens       department document_type    category  region  year  \
0            36  Human Resources  Policy/Legal  Compliance  Global  2020   
1            54  Human Resources  Policy/Legal  Compliance  Global  2021   
2            50  Human Resources  Policy/Legal  Compliance  Global  2022   

    source      file_name  
0  ag_news  ag_news_0.txt  
1  ag_news  ag_news_1.txt  
2  ag_news  ag_news_2.txt  



## Vector Database Setup Explanation

FAISS is used here as the vector database layer for Week 2 because it supports:
- efficient dense vector indexing
- top-K similarity search
- persistent storage through index serialization
- compatibility with embedding pipelines built in notebooks or scripts

The workflow implemented below is:
1. load processed chunk records
2. generate embeddings for each chunk
3. validate dimensional consistency
4. build a FAISS index
5. persist index and metadata for retrieval


In [5]:

model = SentenceTransformer(CONFIG["embedding_model_name"])
texts = df[CONFIG["text_column"]].tolist()

start_time = time.perf_counter()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=CONFIG["normalize_embeddings"]
)
embedding_time = time.perf_counter() - start_time

print("Embeddings shape:", embeddings.shape)
print("Embedding generation time (seconds):", round(embedding_time, 4))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Embeddings shape: (2129, 384)
Embedding generation time (seconds): 209.5474


In [6]:

# dimensional consistency validation
if embeddings.ndim != 2:
    raise ValueError("Embeddings must be a 2D array.")

embedding_dim = embeddings.shape[1]
row_dimensions = [len(vector) for vector in embeddings]
if len(set(row_dimensions)) != 1:
    raise ValueError("Inconsistent embedding dimensions detected.")

print("Validated embedding dimension:", embedding_dim)
print("All embeddings have consistent dimensions.")

# store embeddings and metadata
np.save(CONFIG["vectorstore_dir"] / "embeddings.npy", embeddings)
df.to_csv(CONFIG["vectorstore_dir"] / "metadata_with_chunks.csv", index=False)

manifest = {
    "embedding_model_name": CONFIG["embedding_model_name"],
    "num_vectors": int(embeddings.shape[0]),
    "embedding_dimension": int(embedding_dim),
    "normalized": bool(CONFIG["normalize_embeddings"])
}
with open(CONFIG["vectorstore_dir"] / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Embedding artifacts saved to:", CONFIG["vectorstore_dir"])


Validated embedding dimension: 384
All embeddings have consistent dimensions.
Embedding artifacts saved to: week2_vectorstore


In [7]:

# FAISS index creation and persistence
index = faiss.IndexFlatIP(embedding_dim) if CONFIG["normalize_embeddings"] else faiss.IndexFlatL2(embedding_dim)
index.add(embeddings.astype("float32"))
faiss.write_index(index, str(CONFIG["vectorstore_dir"] / "faiss.index"))

print("FAISS index total vectors:", index.ntotal)
print("FAISS index saved successfully.")


FAISS index total vectors: 2129
FAISS index saved successfully.



## Retrieval Pipeline

The retrieval workflow implemented below performs the required Week 2 steps:
- convert user query to embedding
- retrieve top-K similar chunks
- attach metadata to retrieved results
- support metadata-based filtering before ranking


In [8]:

def apply_metadata_filters(frame, filters=None):
    filtered = frame.copy()
    if not filters:
        return filtered

    for key, value in filters.items():
        if key not in filtered.columns:
            continue
        if isinstance(value, (list, tuple, set)):
            filtered = filtered[filtered[key].isin(value)]
        else:
            filtered = filtered[filtered[key] == value]
    return filtered


def retrieve(query, top_k=None, filters=None):
    top_k = top_k or CONFIG["top_k"]
    filtered_df = apply_metadata_filters(df, filters)

    if filtered_df.empty:
        return pd.DataFrame(columns=["rank", "score"] + df.columns.tolist())

    candidate_indices = filtered_df.index.to_list()
    candidate_embeddings = embeddings[candidate_indices]

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=CONFIG["normalize_embeddings"]
    ).astype("float32")

    if CONFIG["normalize_embeddings"]:
        scores = np.dot(candidate_embeddings, query_embedding[0])
    else:
        scores = cosine_similarity(candidate_embeddings, query_embedding)[..., 0]

    top_positions = np.argsort(scores)[::-1][:top_k]
    top_indices = [candidate_indices[pos] for pos in top_positions]
    top_scores = [float(scores[pos]) for pos in top_positions]

    results = df.loc[top_indices].copy().reset_index(drop=True)
    results.insert(0, "score", top_scores)
    results.insert(0, "rank", range(1, len(results) + 1))
    return results


In [9]:

# example retrieval without metadata filters
sample_query = "compliance policy employee regulation requirements"
results = retrieve(sample_query, top_k=5)
results[["rank", "score", "chunk_id", "source", "year", "file_name", "chunk"]].head(5)


,rank,score,chunk_id,source,year,file_name,chunk
0,1,0.520213,eurlex_123_chunk_1,eurlex,2023,eurlex_123.txt,"COUNCIL REGULATION (EC, EURATOM) No 1873/2006\..."
1,2,0.484961,eurlex_193_chunk_1,eurlex,2023,eurlex_193.txt,Council Regulation (EC) No 334/2002\nof 18 Feb...
2,3,0.483233,eurlex_113_chunk_1,eurlex,2023,eurlex_113.txt,DECISION No 204\nof 6 October 2005\non model f...
3,4,0.480677,eurlex_540_chunk_5,eurlex,2020,eurlex_540.txt,or legal entities and the companies referred t...
4,5,0.468597,eurlex_534_chunk_5,eurlex,2024,eurlex_534.txt,or to prohibit or restrict their being placed ...


In [10]:

# example retrieval with metadata filters
filtered_results = retrieve(
    query="legal compliance policy and requirements",
    top_k=5,
    filters={
        "source": "eurlex",
        "year": 2024,
        "category": "Compliance"
    }
)
filtered_results[["rank", "score", "chunk_id", "source", "year", "file_name", "chunk"]].head(5)


,rank,score,chunk_id,source,year,file_name,chunk
0,1,0.503737,eurlex_534_chunk_5,eurlex,2024,eurlex_534.txt,or to prohibit or restrict their being placed ...
1,2,0.484208,eurlex_534_chunk_8,eurlex,2024,eurlex_534.txt,the main provisions of domestic law which they...
2,3,0.480471,eurlex_284_chunk_6,eurlex,2024,eurlex_284.txt,Regulation (EC) No 955/2005 is hereby amended ...
3,4,0.472766,eurlex_54_chunk_4,eurlex,2024,eurlex_54.txt,compliance with the requirements set out in An...
4,5,0.450753,eurlex_489_chunk_5,eurlex,2024,eurlex_489.txt,"paragraph 1, that one or more of such conditio..."



## Retrieval Evaluation and Performance Analysis

Week 2 requires evaluation metrics and latency tracking.

The evaluation below measures:
- **Hit@K**: whether the correct chunk appears in the top-K results
- **MRR**: reciprocal rank of the correct chunk
- **Average retrieval latency**

To avoid introducing outside datasets or manual labels, the evaluation queries are derived from existing chunk text already present in the processed metadata file. Each sampled query uses a short fragment from the same chunk, and the expected target is that chunk's own `chunk_id`.


In [11]:

rng = np.random.default_rng(CONFIG["random_state"])
sample_size = min(CONFIG["evaluation_sample_size"], len(df))
eval_indices = rng.choice(df.index.to_numpy(), size=sample_size, replace=False)

def build_eval_query(text, max_words=12):
    words = str(text).split()
    return " ".join(words[:max_words])


def evaluate_retrieval(eval_frame, top_k=None):
    top_k = top_k or CONFIG["top_k"]
    records = []

    for idx, row in eval_frame.iterrows():
        query = build_eval_query(row[CONFIG["text_column"]])
        expected_id = row[CONFIG["id_column"]]

        start = time.perf_counter()
        retrieved = retrieve(query, top_k=top_k)
        latency_ms = (time.perf_counter() - start) * 1000

        retrieved_ids = retrieved[CONFIG["id_column"]].tolist()
        hit = int(expected_id in retrieved_ids)

        if expected_id in retrieved_ids:
            rank = retrieved_ids.index(expected_id) + 1
            reciprocal_rank = 1.0 / rank
        else:
            rank = None
            reciprocal_rank = 0.0

        records.append({
            "expected_chunk_id": expected_id,
            "query": query,
            "hit_at_k": hit,
            "rank": rank,
            "reciprocal_rank": reciprocal_rank,
            "latency_ms": latency_ms
        })

    result_df = pd.DataFrame(records)
    summary = {
        "sample_size": int(len(result_df)),
        "top_k": int(top_k),
        "hit_rate_at_k": float(result_df["hit_at_k"].mean()),
        "mrr": float(result_df["reciprocal_rank"].mean()),
        "avg_latency_ms": float(result_df["latency_ms"].mean()),
        "max_latency_ms": float(result_df["latency_ms"].max()),
        "min_latency_ms": float(result_df["latency_ms"].min())
    }
    return result_df, summary


eval_frame = df.loc[eval_indices].reset_index(drop=True)
eval_results_df, eval_summary = evaluate_retrieval(eval_frame, top_k=CONFIG["top_k"])

print(pd.Series(eval_summary))
eval_results_df.head()


sample_size       100.000000
top_k               5.000000
hit_rate_at_k       0.580000
mrr                 0.502000
avg_latency_ms     39.564054
max_latency_ms     94.007323
min_latency_ms     27.628115
dtype: float64


,expected_chunk_id,query,hit_at_k,rank,reciprocal_rank,latency_ms
0,eurlex_543_chunk_4,who have obtained import rights. Licences shal...,0,NaN,0.0,29.785512
1,eurlex_511_chunk_3,"to the actual grant received by the company, c...",1,2.0,0.5,36.653291
2,eurlex_289_chunk_2,"the full year 1999, [...]* for GE and [...]* f...",0,NaN,0.0,40.551185
3,eurlex_165_chunk_1,COMMISSION REGULATION (EEC) No 966/77 of 4 May...,0,NaN,0.0,37.090918
4,eurlex_151_chunk_2,of the CoC’s discussions to the PSC. Article 6...,1,1.0,1.0,39.967630


In [12]:

# filtered retrieval evaluation example
filtered_eval_frame = df[df["source"] == "eurlex"].copy()
filtered_eval_frame = filtered_eval_frame.sample(
    n=min(25, len(filtered_eval_frame)),
    random_state=CONFIG["random_state"]
).reset_index(drop=True)

filtered_records = []
for _, row in filtered_eval_frame.iterrows():
    query = build_eval_query(row[CONFIG["text_column"]])
    expected_id = row[CONFIG["id_column"]]
    filters = {"source": row["source"], "year": row["year"]}

    start = time.perf_counter()
    retrieved = retrieve(query, top_k=CONFIG["top_k"], filters=filters)
    latency_ms = (time.perf_counter() - start) * 1000

    retrieved_ids = retrieved[CONFIG["id_column"]].tolist()
    hit = int(expected_id in retrieved_ids)
    filtered_records.append({
        "expected_chunk_id": expected_id,
        "filters": str(filters),
        "hit_at_k": hit,
        "latency_ms": latency_ms
    })

filtered_eval_results = pd.DataFrame(filtered_records)
filtered_eval_summary = {
    "sample_size": int(len(filtered_eval_results)),
    "filtered_hit_rate_at_k": float(filtered_eval_results["hit_at_k"].mean()) if not filtered_eval_results.empty else None,
    "filtered_avg_latency_ms": float(filtered_eval_results["latency_ms"].mean()) if not filtered_eval_results.empty else None
}

print(pd.Series(filtered_eval_summary))
filtered_eval_results.head()


sample_size                25.000000
filtered_hit_rate_at_k      0.800000
filtered_avg_latency_ms    74.108633
dtype: float64


,expected_chunk_id,filters,hit_at_k,latency_ms
0,eurlex_223_chunk_1,"{'source': 'eurlex', 'year': 2023}",0,23.189853
1,eurlex_321_chunk_1,"{'source': 'eurlex', 'year': 2021}",0,26.126159
2,eurlex_511_chunk_20,"{'source': 'eurlex', 'year': 2021}",1,21.078215
3,eurlex_483_chunk_14,"{'source': 'eurlex', 'year': 2023}",1,21.691868
4,eurlex_220_chunk_43,"{'source': 'eurlex', 'year': 2020}",1,21.115170


In [13]:

# save evaluation artifacts
outputs = {
    "embedding_generation_seconds": embedding_time,
    "retrieval_evaluation": eval_summary,
    "filtered_retrieval_evaluation": filtered_eval_summary
}

with open(CONFIG["vectorstore_dir"] / "evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2)

eval_results_df.to_csv(CONFIG["vectorstore_dir"] / "retrieval_evaluation_details.csv", index=False)
filtered_eval_results.to_csv(CONFIG["vectorstore_dir"] / "filtered_retrieval_evaluation_details.csv", index=False)

print("Saved:")
print("-", CONFIG["vectorstore_dir"] / "evaluation_summary.json")
print("-", CONFIG["vectorstore_dir"] / "retrieval_evaluation_details.csv")
print("-", CONFIG["vectorstore_dir"] / "filtered_retrieval_evaluation_details.csv")


Saved:
- week2_vectorstore/evaluation_summary.json
- week2_vectorstore/retrieval_evaluation_details.csv
- week2_vectorstore/filtered_retrieval_evaluation_details.csv


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



## Performance Observations

Use the generated evaluation outputs to document the following in your submission:
- observed embedding generation time
- average retrieval latency
- filtered vs unfiltered retrieval behavior
- any bottlenecks noticed when increasing candidate size or filter scope

This notebook completes the Week 2 requirements and prepares the system for the next phase: LLM integration.
